# Time series EDA

Change `TICKER` and rerun for single-stock analysis. The cross-sectional
sections use the full universe.

In [ ]:
TICKER = "SPY"
START, END = None, None

In [ ]:
import datetime as dt

import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots
from scipy import stats

from sdp import dal

con = dal.con()
bars = dal.day_aggs(START, END)
print(dal.status())

---
## 1. Adjusted price series

In [ ]:
import duckdb
from sdp.config import settings

wh = duckdb.connect(str(settings.warehouse_path), read_only=True)

prices = wh.sql(f"""
    select p.date, p.close, p.volume, p.adj_close_split, p.adj_close_total
    from main_staging.stg_prices_adjusted p
    where p.ticker = '{TICKER}'
    order by p.date
""").pl()

print(f"{TICKER}: {len(prices)} sessions, {prices['date'].min()} to {prices['date'].max()}")

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f"{TICKER} unadjusted close", f"{TICKER} adjusted close (split + dividend)",
])
fig.add_trace(go.Scatter(x=prices["date"], y=prices["close"], mode="lines",
                         line=dict(width=1), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=prices["date"], y=prices["adj_close_total"], mode="lines",
                         line=dict(width=1), showlegend=False), row=1, col=2)
fig.update_yaxes(title_text="$", row=1, col=1)
fig.update_yaxes(title_text="$", row=1, col=2)
fig.update_layout(height=350, margin=dict(t=40, b=30))
fig.show()

---
## 2. Log returns

In [ ]:
ret = (
    prices.filter(pl.col("adj_close_total").is_not_null())
    .with_columns(
        (pl.col("adj_close_total") / pl.col("adj_close_total").shift(1))
        .log()
        .alias("log_ret"),
    )
    .filter(pl.col("log_ret").is_not_null())
)

print(f"{len(ret)} return observations")

px.line(ret, x="date", y="log_ret", title=f"{TICKER} daily log returns",
        labels={"log_ret": "log return", "date": ""},
        height=300).update_traces(line_width=0.5)

---
## 3. Return distribution

Histogram against a fitted normal. Equity returns are heavier-tailed than
normal (positive excess kurtosis).

In [ ]:
x = ret["log_ret"].to_numpy()
mu, sigma = x.mean(), x.std(ddof=1)

xgrid = np.linspace(x.min(), x.max(), 300)

fig = go.Figure()
fig.add_trace(go.Histogram(x=x, nbinsx=80, histnorm="probability density",
                           opacity=0.6, name="log returns"))
fig.add_trace(go.Scatter(x=xgrid, y=stats.norm.pdf(xgrid, mu, sigma),
                         mode="lines", line=dict(color="red", width=1.5),
                         name="normal fit"))
fig.update_layout(title=f"{TICKER} log return distribution vs normal",
                  xaxis_title="log return", height=400, margin=dict(t=40, b=30))
fig.show()

---
## 4. Summary statistics

In [ ]:
ann = 252
jb_p = stats.jarque_bera(x).pvalue

pl.DataFrame({
    "statistic": [
        "observations", "mean (daily)", "mean (ann.)", "std (daily)", "std (ann.)",
        "skewness", "excess kurtosis", "min", "max", "Jarque-Bera p",
    ],
    "value": [
        f"{len(x)}", f"{mu:.6f}", f"{mu * ann:.4f}", f"{sigma:.6f}",
        f"{sigma * np.sqrt(ann):.4f}", f"{stats.skew(x):.4f}",
        f"{stats.kurtosis(x):.4f}", f"{x.min():.4f}", f"{x.max():.4f}",
        f"{jb_p:.6f}",
    ],
})

---
## 5. QQ plot: raw returns vs GARCH(1,1) residuals

If volatility clustering explains the fat tails, the standardised residuals
from a GARCH(1,1) should sit closer to the normal line than the raw returns.

In [ ]:
from arch import arch_model

z_raw = (x - mu) / sigma

am = arch_model(x * 100, vol="Garch", p=1, q=1, mean="Constant")
res = am.fit(disp="off")
z_garch = res.std_resid

osm_raw, osr_raw = stats.probplot(z_raw, dist="norm", fit=False)
osm_garch, osr_garch = stats.probplot(z_garch, dist="norm", fit=False)

lo = min(osm_raw.min(), osm_garch.min())
hi = max(osm_raw.max(), osm_garch.max())

fig = go.Figure()
fig.add_trace(go.Scatter(x=osm_raw, y=osr_raw, mode="markers",
                         marker=dict(size=3, opacity=0.4, color="steelblue"),
                         name="raw returns"))
fig.add_trace(go.Scatter(x=osm_garch, y=osr_garch, mode="markers",
                         marker=dict(size=3, opacity=0.4, color="darkorange"),
                         name="GARCH(1,1) residuals"))
fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines",
                         line=dict(color="red", dash="dash"), name="normal"))
fig.update_layout(title=f"{TICKER} QQ plot: raw vs GARCH(1,1) residuals",
                  xaxis_title="normal quantiles", yaxis_title="sample quantiles",
                  height=500, width=550, margin=dict(t=40, b=30))
fig.show()

print(f"GARCH(1,1) parameters:  ω = {res.params['omega']:.4f}, "
      f"α = {res.params['alpha[1]']:.4f}, β = {res.params['beta[1]']:.4f}")
print(f"Raw returns excess kurtosis:       {stats.kurtosis(z_raw):.3f}")
print(f"GARCH residuals excess kurtosis:   {stats.kurtosis(z_garch):.3f}")

---
## 6. Autocorrelation

Returns should show near-zero ACF. Absolute returns should show persistence
(volatility clustering).

In [ ]:
from statsmodels.tsa.stattools import acf

max_lag = 40
acf_ret = acf(x, nlags=max_lag, fft=True)
acf_abs = acf(np.abs(x), nlags=max_lag, fft=True)
ci = 1.96 / np.sqrt(len(x))
lags = list(range(1, max_lag + 1))

fig = make_subplots(rows=1, cols=2, subplot_titles=["ACF of log returns", "ACF of |log returns|"])
fig.add_trace(go.Bar(x=lags, y=acf_ret[1:], showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=lags, y=acf_abs[1:], showlegend=False), row=1, col=2)
for col in (1, 2):
    fig.add_hline(y=ci, line_dash="dash", line_color="grey", row=1, col=col)
    fig.add_hline(y=-ci, line_dash="dash", line_color="grey", row=1, col=col)
fig.update_layout(height=300, title=f"{TICKER} autocorrelation", margin=dict(t=60, b=30))
fig.show()

---
## 7. AR(1)

$x_t = c + \phi\, x_{t-1} + \varepsilon_t$. For a liquid name, $|\phi|$
should be small.

In [ ]:
from statsmodels.tsa.ar_model import AutoReg

model = AutoReg(x, lags=1).fit()

print(f"c   = {model.params[0]:.6f}")
print(f"phi = {model.params[1]:.6f}")
print(f"AIC = {model.aic:.2f}")
print(f"BIC = {model.bic:.2f}")
print()
print(model.summary().tables[1])

---
## 8. Rolling volatility

20-day and 60-day rolling standard deviation (annualised).

In [ ]:
vol = (
    ret.with_columns(
        (pl.col("log_ret").rolling_std(20) * np.sqrt(252)).alias("vol_20d"),
        (pl.col("log_ret").rolling_std(60) * np.sqrt(252)).alias("vol_60d"),
    )
    .filter(pl.col("vol_60d").is_not_null())
    .select("date", "vol_20d", "vol_60d")
    .unpivot(index="date", variable_name="window", value_name="volatility")
)

px.line(vol, x="date", y="volatility", color="window",
        title=f"{TICKER} rolling annualised volatility",
        labels={"volatility": "σ (ann.)", "date": ""},
        height=300)

---
## 9. Cross-sectional returns

Load the universe and compute daily log returns across all tickers.

In [ ]:
cs = wh.sql("""
    select p.ticker, p.date, p.adj_close_total,
           ln(p.adj_close_total
              / lag(p.adj_close_total) over (partition by p.ticker order by p.date))
               as log_ret
    from main_staging.stg_prices_adjusted p
    join main_staging.stg_universe u using (ticker, date)
    where u.in_universe and p.adj_close_total is not null
    qualify log_ret is not null
    order by p.date, p.ticker
""").pl()

n_dates = cs["date"].n_unique()
n_tickers = cs["ticker"].n_unique()
print(f"{len(cs):,} rows, {n_tickers:,} tickers, {n_dates:,} dates")

---
## 10. Cross-sectional dispersion

Standard deviation of returns across tickers on each date. Higher dispersion
means more room for stock selection.

In [ ]:
dispersion = cs.group_by("date").agg(
    pl.col("log_ret").std().alias("cs_std"),
    pl.col("log_ret").mean().alias("cs_mean"),
    pl.col("ticker").n_unique().alias("n_tickers"),
).sort("date")

px.line(dispersion, x="date", y="cs_std",
        title="Cross-sectional return dispersion (daily)",
        labels={"cs_std": "std dev of returns", "date": ""},
        height=300)

---
## 11. Cross-sectional distribution on a single date

Pick a date and look at the return distribution across all tickers that day.

In [ ]:
SAMPLE_DATE = cs["date"].max()

one_day = cs.filter(pl.col("date") == SAMPLE_DATE)
print(f"{SAMPLE_DATE}: {len(one_day)} tickers")

px.histogram(one_day, x="log_ret", nbins=60,
             title=f"Cross-sectional log returns on {SAMPLE_DATE} ({len(one_day)} tickers)",
             labels={"log_ret": "log return"},
             height=350)

---
## 12. Pairwise correlations

Correlation of daily returns for the most liquid names.

In [ ]:
N_CORR = 20

top = wh.sql(f"""
    select p.ticker, avg(p.close * p.volume) as adv
    from main_staging.stg_prices_adjusted p
    join main_staging.stg_universe u using (ticker, date)
    where u.in_universe and p.adj_close_total is not null
    group by p.ticker
    order by adv desc
    limit {N_CORR}
""").pl()["ticker"].to_list()

wide = (
    cs.filter(pl.col("ticker").is_in(top))
    .pivot(on="ticker", index="date", values="log_ret")
    .drop("date")
)

corr = wide.to_pandas()[top].corr()

fig = px.imshow(corr, x=top, y=top, color_continuous_scale="RdBu_r",
                zmin=-1, zmax=1, aspect="equal",
                title=f"Pairwise return correlation, top {N_CORR} by dollar volume",
                labels=dict(color="ρ"))
fig.update_layout(height=550, width=600, margin=dict(t=40, b=30))
fig.show()

---
## 13. Cumulative returns comparison

A few tickers side by side. Indexed to 1 at the start of the window.

In [ ]:
COMPARE = ["AAPL", "MSFT", "GOOGL", "AMZN", "JPM"]

cum = (
    cs.filter(pl.col("ticker").is_in(COMPARE))
    .sort("ticker", "date")
    .with_columns(
        pl.col("log_ret").cum_sum().over("ticker").exp().alias("indexed"),
    )
)

px.line(cum, x="date", y="indexed", color="ticker",
        title="Cumulative return (indexed to 1)",
        labels={"indexed": "indexed return", "date": ""},
        height=350)

---
## 14. GARCH(1,1) QQ across tickers

Same raw-vs-GARCH QQ comparison for a panel of tickers. Shows whether the
improvement from GARCH is consistent across names.

In [ ]:
QQ_TICKERS = ["AAPL", "MSFT", "JPM", "XOM", "TSLA", "META"]

n_cols = 3
n_rows = (len(QQ_TICKERS) + n_cols - 1) // n_cols

fig = make_subplots(rows=n_rows, cols=n_cols,
                    subplot_titles=QQ_TICKERS,
                    horizontal_spacing=0.06, vertical_spacing=0.1)

kurtosis_rows = []

for i, tk in enumerate(QQ_TICKERS):
    r, c = divmod(i, n_cols)
    r += 1; c += 1

    tk_ret = wh.sql(f"""
        select ln(p.adj_close_total
                  / lag(p.adj_close_total) over (order by p.date)) as log_ret
        from main_staging.stg_prices_adjusted p
        join main_staging.stg_universe u using (ticker, date)
        where p.ticker = '{tk}' and u.in_universe and p.adj_close_total is not null
        qualify log_ret is not null
        order by p.date
    """).pl()["log_ret"].to_numpy()

    mu_tk, sig_tk = tk_ret.mean(), tk_ret.std(ddof=1)
    z_raw_tk = (tk_ret - mu_tk) / sig_tk

    am_tk = arch_model(tk_ret * 100, vol="Garch", p=1, q=1, mean="Constant")
    res_tk = am_tk.fit(disp="off")
    z_garch_tk = res_tk.std_resid

    osm_r, osr_r = stats.probplot(z_raw_tk, dist="norm", fit=False)
    osm_g, osr_g = stats.probplot(z_garch_tk, dist="norm", fit=False)

    show_legend = (i == 0)
    fig.add_trace(go.Scatter(x=osm_r, y=osr_r, mode="markers",
                             marker=dict(size=2, opacity=0.3, color="steelblue"),
                             name="raw", showlegend=show_legend,
                             legendgroup="raw"), row=r, col=c)
    fig.add_trace(go.Scatter(x=osm_g, y=osr_g, mode="markers",
                             marker=dict(size=2, opacity=0.3, color="darkorange"),
                             name="GARCH", showlegend=show_legend,
                             legendgroup="garch"), row=r, col=c)

    lo = min(osm_r.min(), osm_g.min())
    hi = max(osm_r.max(), osm_g.max())
    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines",
                             line=dict(color="red", dash="dash", width=1),
                             showlegend=False), row=r, col=c)

    kurtosis_rows.append({
        "ticker": tk,
        "n": len(tk_ret),
        "raw_kurt": round(stats.kurtosis(z_raw_tk), 2),
        "garch_kurt": round(stats.kurtosis(z_garch_tk), 2),
        "alpha": round(res_tk.params["alpha[1]"], 4),
        "beta": round(res_tk.params["beta[1]"], 4),
    })

fig.update_layout(height=250 * n_rows, width=900,
                  title="QQ plots: raw returns vs GARCH(1,1) residuals",
                  margin=dict(t=60, b=30))
fig.show()

pl.DataFrame(kurtosis_rows)

---
## 15. Universe size over time

In [ ]:
px.line(dispersion, x="date", y="n_tickers",
        title="In-universe ticker count per date",
        labels={"n_tickers": "count", "date": ""},
        height=250)